In [ ]:
#### AVA: A Large-Scale Database for Aesthetic Visual Analysis ####
#### Get aesthetic_image_lists ####
import os
import pandas as pd
base_image_list_dir = "/data_center/data2/dataset/chenwy/21164-data/dpo_dataset/aesthetic_visual_analysis/AVA_dataset/aesthetics_image_lists"
output_csv_file_path = "/data_center/data2/dataset/chenwy/21164-data/dpo_dataset/aesthetic_visual_analysis/AVA_dataset/aesthetic_image_uids.csv"

all_uids = set()

for filename in os.listdir(base_image_list_dir):
    if filename.endswith('.jpgl'):
        file_path = os.path.join(base_image_list_dir, filename)
        print(f"Processing {filename}...")
        
        with open(file_path, 'r') as f:
            for line in f:
                uid = line.strip()
                if uid:  # 确保不是空行
                    all_uids.add(uid)

uids_list = sorted(list(all_uids), key=lambda x: int(x))
print(f"\nTotal unique UIDs found: {len(uids_list)}")
pd.DataFrame({'uid': uids_list}).to_csv(output_csv_file_path, index=False)

In [ ]:
#### Score Distribution ####
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

image_uid_csv_path = "/data_center/data2/dataset/chenwy/21164-data/dpo_dataset/aesthetic_visual_analysis/AVA_dataset/aesthetic_image_uids.csv"
score_information_path = "/data_center/data2/dataset/chenwy/21164-data/dpo_dataset/aesthetic_visual_analysis/AVA_dataset/AVA.txt"

image_uids_df = pd.read_csv(image_uid_csv_path)
target_uids = set(image_uids_df['uid'].astype(str))

print(f"Total target UIDs: {len(target_uids)}")

# Load score information from AVA.txt
# Column 1: Index
# Column 2: Image ID
# Columns 3-12: Counts of ratings 1-10
scores_dict = {}

print("Reading AVA.txt...")
with open(score_information_path, 'r') as f:
    for line in f:
        parts = line.strip().split()
        if len(parts) >= 12:
            image_id = parts[1]  # Column 2: Image ID
            if image_id in target_uids:
                # Columns 3-12: Counts of ratings 1-10
                rating_counts = [int(parts[i]) for i in range(2, 12)]  # indices 2-11 (0-indexed)
                
                total_ratings = sum(rating_counts)
                if total_ratings > 0:
                    weighted_sum = sum((i + 1) * count for i, count in enumerate(rating_counts))
                    average_score = weighted_sum / total_ratings
                    scores_dict[image_id] = average_score

print(f"Found scores for {len(scores_dict)} images")

scores_df = pd.DataFrame({
    'uid': list(scores_dict.keys()),
    'average_score': list(scores_dict.values())
})

print(f"\nScore Statistics:")
print(scores_df['average_score'].describe())

scores_above_5 = scores_df[scores_df['average_score'] > 5]
count_above_5 = len(scores_above_5)
percentage_above_5 = (count_above_5 / len(scores_df)) * 100
print(f"\nImages with score > 5: {count_above_5} ({percentage_above_5:.2f}%)")

plt.figure(figsize=(12, 6),dpi=300)
plt.hist(scores_df['average_score'], bins=50, edgecolor='black', alpha=0.7)
plt.xlabel('Average Score', fontsize=12)
plt.ylabel('Frequency', fontsize=12)
plt.title('Distribution of Average Aesthetic Scores', fontsize=14)
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

output_scores_path = image_uid_csv_path.replace('.csv', '_with_scores.csv')
scores_df.to_csv(output_scores_path, index=False)

In [5]:
#### Get high-aesthetic images ####
import os
import pandas as pd
from tqdm import tqdm
source_image_dir = "/data_center/data2/dataset/chenwy/21164-data/dpo_dataset/aesthetic_visual_analysis/AVA_dataset/image"
target_image_dir = "/data_center/data2/dataset/chenwy/21164-data/dpo_dataset/aesthetic_visual_analysis/AVA_dataset/high_aesthetic_images"
os.makedirs(target_image_dir, exist_ok=True)
image_uid_with_scores_csv_path = "/data_center/data2/dataset/chenwy/21164-data/dpo_dataset/aesthetic_visual_analysis/AVA_dataset/aesthetic_image_uids_with_scores.csv"

image_uids_with_scores_df = pd.read_csv(image_uid_with_scores_csv_path)

for index, row in tqdm(image_uids_with_scores_df.iterrows()):
    uid = int(row['uid'])
    score = row['average_score']
    if score > 5:
        source_image_path = os.path.join(source_image_dir, f"{uid}.jpg")
        target_image_path = os.path.join(target_image_dir, f"{uid}.jpg")
        os.symlink(source_image_path, target_image_path)


0it [00:00, ?it/s]

39854it [00:21, 1834.86it/s]
